# Fashion MNIST - Liverpool ML assignment

Open this notebook in Colab via `https://colab.research.google.com/github/Cyberwookie69/fashion-mnist/blob/main/Fashion_MNIST_Colab.ipynb`.

This notebook clones the repo and runs the scripts as-is.

## 1. Setup - clone repo and install deps

Skipped automatically if you're already inside the cloned repo (e.g. running locally).

In [ ]:
import os
from pathlib import Path

REPO = 'fashion-mnist'
if not Path(REPO).exists() and not (Path.cwd().name == REPO):
    !git clone https://github.com/Cyberwookie69/{REPO}.git
if Path(REPO).exists() and Path.cwd().name != REPO:
    os.chdir(REPO)
print('Working dir:', Path.cwd())
print('Files:', sorted(p.name for p in Path.cwd().iterdir() if p.is_file())[:15])

In [ ]:
!pip install -q tqdm

## 2. Patch settings for Colab

Colab gives 2 vCPUs and uses Linux paths. The scripts hardcode 6 workers and a Windows path; this cell rewrites them in-memory before we run.

In [ ]:
import re
for py in Path('.').glob('*.py'):
    text = py.read_text(encoding='utf-8')
    new = text.replace(r'r"c:\temp\temp\fashion"', f'r"{Path.cwd()}"')
    new = re.sub(r'NUM_WORKERS\s*=\s*6', 'NUM_WORKERS = 2', new)
    new = re.sub(r'INTRA_OP\s*=\s*2', 'INTRA_OP = 1', new)
    if new != text:
        py.write_text(new, encoding='utf-8')
        print(f'patched {py.name}')

## 3. Main run - Parts 1, 2, 3 plus extensions

Takes ~10-20 minutes on Colab CPU, ~3-5 minutes with T4 GPU enabled.

In [ ]:
!python fashion_mnist.py

In [ ]:
from IPython.display import Image, Markdown, display
for png in ['samples_preview.png', 'training_curves.png', 'confusion_matrices.png',
            'gradient_norms_per_layer.png', 'lr_sweep_plot.png', 'pareto_acc_vs_time.png']:
    if Path(png).exists():
        display(Markdown(f'### {png}'))
        display(Image(png))

In [ ]:
from IPython.display import Markdown
if Path('results_summary.md').exists():
    display(Markdown(Path('results_summary.md').read_text(encoding='utf-8')))

## 4. Matrix sweep results (pre-computed)

The full sweep is 1300 cells / ~2 hours locally. The repo ships with a pre-computed `matrix_results.csv` so Colab does not have to run it again. The cell below just runs `analyze()` to regenerate the heatmaps and `matrix_summary.md` from that CSV.

If you ever delete the CSV and want to re-run it on Colab, uncomment the patch cell below to reduce the grid to 117 cells (~30 min on free Colab).

In [ ]:
# Optional fallback: shrink the grid if matrix_results.csv is missing.
# Safe to leave commented when the CSV ships with the repo.
# import re
# ms_path = Path('matrix_sweep.py')
# ms = ms_path.read_text(encoding='utf-8')
# ms = re.sub(r'EPOCH_GRID = \[.*?\]', 'EPOCH_GRID = [10, 30, 100]', ms)
# ms = re.sub(r'BATCH_GRID = \[.*?\]', 'BATCH_GRID = [32, 256, 2048]', ms)
# ms_path.write_text(ms, encoding='utf-8')
if Path('matrix_results.csv').exists():
    n = sum(1 for _ in open('matrix_results.csv')) - 1
    print(f'matrix_results.csv present ({n} rows). analyze() will skip the sweep.')
else:
    print('matrix_results.csv missing. Uncomment the patch above to mini-sweep.')

In [ ]:
!python matrix_sweep.py

In [ ]:
for png in ['dropout_optimum_heatmap.png', 'matrix_heatmap_test_acc.png',
            'sigmoid_3d.png', 'relu_3d.png']:
    if Path(png).exists():
        display(Markdown(f'### {png}'))
        display(Image(png))
if Path('matrix_summary.md').exists():
    display(Markdown(Path('matrix_summary.md').read_text(encoding='utf-8')))

## 5. Optional - 3D wiremesh plots for sigmoid / ReLU

These read `matrix_results.csv` from the cell above. If the mini-sweep ran, they get only 9 datapoints per activation, so the wireframe is sparse but still shows the pattern.

In [ ]:
!python plot_sigmoid_3d.py
!python plot_relu_3d.py

In [ ]:
for png in ['sigmoid_3d.png', 'relu_3d.png']:
    if Path(png).exists():
        display(Markdown(f'### {png}'))
        display(Image(png))

## 6. Download outputs

Save the generated PNGs and reports to your local machine.

In [ ]:
try:
    from google.colab import files
    for ext in ('*.png', '*.md'):
        for path in Path('.').glob(ext):
            if path.name in ('README.md',):
                continue
            files.download(str(path))
except ImportError:
    print('Not running on Colab - skip download.')